# Mixture-of-Experts: 3-Way Comparison on Kestrel

Proof-of-concept experiment to determine whether training separate XGBoost models
per wallclock cluster improves runtime prediction beyond what a single model with
wallclock_requested as a feature already achieves.

**3-way comparison:**
1. Single model WITHOUT wallclock_requested (baseline)
2. Single model WITH wallclock_requested (current default)
3. Separate expert per wallclock cluster

Only if #3 beats #2 is the extra complexity of mixture-of-experts justified.

**Dataset:** NLR Kestrel, benchmark window 2025-03-29 to 2025-06-26

**Related:** Issue [#124](https://github.com/NatLabRockies/hpc-oda-commons/issues/124)

## 1. Setup

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from hpc_oda_commons.models.job_runtime_xgboost.model import (
    JobRuntimeXGBoostConfig,
    JobRuntimeXGBoostModel,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

REPO_ROOT = Path.cwd().parent.parent
DATA_PATH = REPO_ROOT / 'workspace' / 'data' / 'datasets' / 'nlr_kestrel' / 'data.parquet'

In [ ]:
# Load and slice to benchmark window
table = pq.read_table(DATA_PATH)
lo = datetime(2025, 3, 29, tzinfo=timezone.utc)
hi = datetime(2025, 6, 26, tzinfo=timezone.utc) + timedelta(days=1)
sc = table.column('submit_time')
ec = table.column('end_time')
mask = pc.and_(
    pc.less(sc, pa.scalar(hi, type=sc.type)),
    pc.greater_equal(ec, pa.scalar(lo, type=ec.type)),
)
df = table.filter(mask).to_pandas()

# Use a sample to keep runtime manageable
SAMPLE_SIZE = 30_000
if len(df) > SAMPLE_SIZE:
    df = df.tail(SAMPLE_SIZE).copy()
    print(f'Using last {SAMPLE_SIZE:,} rows of window (out of {table.filter(mask).num_rows:,})')
else:
    print(f'Using all {len(df):,} rows')

rows_full = df.to_dict('records')
print(f'Loaded {len(rows_full):,} rows')

## 2. Define wallclock clusters

Use the spike detection from the wallclock analysis to define natural bin boundaries.
Jobs are assigned to the cluster matching their requested wallclock value.

In [ ]:
# Define bins based on Kestrel's detected partition limits
# Boundaries at 1h, 2h, 4h, 12h, 24h, 48h
BIN_EDGES_H = [0, 1, 2, 4, 12, 24, 48, float('inf')]
BIN_LABELS = ['<1h', '1-2h', '2-4h', '4-12h', '12-24h', '24-48h', '>48h']

df['wc_hours'] = df['requested_seconds'] / 3600
df['wc_bin'] = pd.cut(df['wc_hours'], bins=BIN_EDGES_H, labels=BIN_LABELS)

print('Wallclock clusters:')
print(f'{"Bin":<10} {"Jobs":>8} {"Pct":>6} {"RT median":>10} {"RT mean":>10}')
print('-' * 50)
for label in BIN_LABELS:
    subset = df[df['wc_bin'] == label]
    if len(subset) == 0:
        continue
    rt = subset['runtime_seconds'].dropna()
    print(f'{label:<10} {len(subset):>8,} {len(subset)/len(df)*100:>5.1f}% '
          f'{rt.median():>10,.0f}s {rt.mean():>10,.0f}s')

## 3. Comparison 1: Single model WITHOUT wallclock

Remove `requested_seconds` from the rows before evaluation.
The model can only use partition, user, cores, memory, etc.

In [ ]:
# Remove wallclock from the data
rows_no_wc = [{k: v for k, v in row.items() if k != 'requested_seconds'} for row in rows_full]

config_shared = dict(
    n_windows=4,
    test_window_hours=6,
    training_lookback_days=30,
    max_svd_components=16,
    target_max_one_hot_width=128,
    random_state=42,
)

print('Running comparison 1: single model WITHOUT wallclock...')
model_no_wc = JobRuntimeXGBoostModel(JobRuntimeXGBoostConfig(**config_shared))
payload_no_wc = model_no_wc.evaluate(rows_no_wc)

print(f'  MAE:  {payload_no_wc["mae"]:,.1f}s')
print(f'  RMSE: {payload_no_wc["rmse"]:,.1f}s')
print(f'  Scored: {payload_no_wc["summary"]["rows_scored"]:,} rows')

## 4. Comparison 2: Single model WITH wallclock

The current default — `requested_seconds` is available as a numeric feature.
XGBoost can split on it internally.

In [ ]:
print('Running comparison 2: single model WITH wallclock...')
model_with_wc = JobRuntimeXGBoostModel(JobRuntimeXGBoostConfig(**config_shared))
payload_with_wc = model_with_wc.evaluate(rows_full)

print(f'  MAE:  {payload_with_wc["mae"]:,.1f}s')
print(f'  RMSE: {payload_with_wc["rmse"]:,.1f}s')
print(f'  Scored: {payload_with_wc["summary"]["rows_scored"]:,} rows')

## 5. Comparison 3: Separate experts per wallclock cluster

Split the data into bins by `requested_seconds`, train a separate XGBoost
model on each bin, then combine the predictions.

Each expert sees only jobs from its cluster. Routing is deterministic:
given `requested_seconds`, the job goes to the bin containing that value.

In [ ]:
print('Running comparison 3: separate experts per wallclock cluster...\n')

# Add bin label to each row
bin_edges_s = [e * 3600 for e in BIN_EDGES_H]

def assign_bin(wc_seconds):
    """Assign a wallclock value to its bin label."""
    for i in range(len(bin_edges_s) - 1):
        if wc_seconds <= bin_edges_s[i + 1]:
            return BIN_LABELS[i]
    return BIN_LABELS[-1]

# Group rows by bin
bin_rows = {label: [] for label in BIN_LABELS}
for row in rows_full:
    wc = row.get('requested_seconds')
    if wc is None or wc <= 0:
        continue
    label = assign_bin(wc)
    bin_rows[label].append(row)

# Train and evaluate each expert
expert_results = {}
all_y_true = []
all_y_pred = []

for label in BIN_LABELS:
    rows_bin = bin_rows[label]
    if len(rows_bin) < 50:
        print(f'  {label}: only {len(rows_bin)} rows — skipping (too few for rolling eval)')
        continue
    
    # Remove wallclock from expert's features — it's now redundant
    # since all jobs in this bin have similar wallclock values
    rows_bin_no_wc = [{k: v for k, v in r.items() if k != 'requested_seconds'} for r in rows_bin]
    
    try:
        model = JobRuntimeXGBoostModel(JobRuntimeXGBoostConfig(**config_shared))
        payload = model.evaluate(rows_bin_no_wc)
        scored = payload['summary']['rows_scored']
        
        if scored == 0:
            print(f'  {label}: {len(rows_bin):,} rows but 0 scored — rolling windows found no valid splits')
            continue
        
        expert_results[label] = payload
        print(f'  {label}: {len(rows_bin):,} rows, scored={scored:,}, '
              f'MAE={payload["mae"]:,.1f}s, RMSE={payload["rmse"]:,.1f}s')
        
        # Collect predictions for global metric computation
        if '_y_true' in payload and '_y_pred' in payload:
            all_y_true.extend(payload['_y_true'])
            all_y_pred.extend(payload['_y_pred'])
    except ValueError as e:
        print(f'  {label}: {len(rows_bin):,} rows — failed: {e}')

# Compute combined metrics (correct RMSE aggregation: combine raw squared errors)
if all_y_true:
    combined_mae = np.mean(np.abs(np.array(all_y_true) - np.array(all_y_pred)))
    combined_rmse = np.sqrt(np.mean((np.array(all_y_true) - np.array(all_y_pred))**2))
    print(f'\n  COMBINED (all experts): MAE={combined_mae:,.1f}s, RMSE={combined_rmse:,.1f}s, '
          f'total scored={len(all_y_true):,}')
else:
    # Fallback: weighted average from per-expert metrics
    total_scored = sum(r['summary']['rows_scored'] for r in expert_results.values())
    if total_scored > 0:
        combined_mae = sum(r['mae'] * r['summary']['rows_scored'] for r in expert_results.values()) / total_scored
        combined_rmse = None  # Can't properly aggregate RMSE from per-bin values
        print(f'\n  COMBINED (weighted MAE): MAE={combined_mae:,.1f}s, total scored={total_scored:,}')
        print('  Note: RMSE cannot be properly aggregated from per-bin values')

## 6. Results Comparison

In [ ]:
print('=' * 70)
print('3-WAY COMPARISON RESULTS — NLR KESTREL')
print('=' * 70)

mae_no_wc = payload_no_wc['mae']
mae_with_wc = payload_with_wc['mae']

print(f'\n{"Approach":<45} {"MAE":>10} {"RMSE":>10} {"vs baseline":>12}')
print('-' * 80)
print(f'{"1. No wallclock (baseline)":<45} {mae_no_wc:>10,.1f}s {payload_no_wc["rmse"]:>10,.1f}s {"—":>12}')

wc_change = (mae_with_wc - mae_no_wc) / mae_no_wc * 100
print(f'{"2. With wallclock (current default)":<45} {mae_with_wc:>10,.1f}s {payload_with_wc["rmse"]:>10,.1f}s {wc_change:>+11.1f}%')

if all_y_true:
    expert_change = (combined_mae - mae_no_wc) / mae_no_wc * 100
    expert_vs_single = (combined_mae - mae_with_wc) / mae_with_wc * 100
    print(f'{"3. Separate experts per cluster":<45} {combined_mae:>10,.1f}s {combined_rmse:>10,.1f}s {expert_change:>+11.1f}%')
    print(f'\n  Expert vs single-with-wallclock: {expert_vs_single:+.1f}%')
    if expert_vs_single < -5:
        print('  --> Mixture of experts IMPROVES over single model')
    elif expert_vs_single > 5:
        print('  --> Single model is BETTER than mixture of experts')
    else:
        print('  --> Results are SIMILAR — mixture complexity may not be justified')

In [ ]:
# Visualize
approaches = ['No wallclock\n(baseline)', 'With wallclock\n(current)', 'Separate\nexperts']
maes = [mae_no_wc, mae_with_wc]
if all_y_true:
    maes.append(combined_mae)
else:
    approaches = approaches[:2]

colors = ['steelblue', 'coral', 'seagreen']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(approaches[:len(maes)], maes, color=colors[:len(maes)])

for bar, mae in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{mae:,.0f}s', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('MAE (seconds, lower = better)')
ax.set_title('3-Way Comparison: Does Mixture of Experts Help?\nNLR Kestrel, benchmark window')
plt.tight_layout()
plt.show()

## 7. Per-Expert Analysis

How does each expert perform on its own cluster? Some clusters may be
much easier to predict than others.

In [ ]:
if expert_results:
    print(f'{"Cluster":<12} {"Jobs":>8} {"Scored":>8} {"MAE":>10} {"RMSE":>10} {"RT median":>10}')
    print('-' * 65)
    for label in BIN_LABELS:
        if label not in expert_results:
            continue
        p = expert_results[label]
        n_bin = len(bin_rows[label])
        rt_med = np.median([r['runtime_seconds'] for r in bin_rows[label] if r.get('runtime_seconds')])
        print(f'{label:<12} {n_bin:>8,} {p["summary"]["rows_scored"]:>8,} '
              f'{p["mae"]:>10,.1f}s {p["rmse"]:>10,.1f}s {rt_med:>10,.0f}s')
else:
    print('No expert results available.')

## 8. Conclusions

In [ ]:
print('CONCLUSIONS')
print('=' * 60)
print()
print(f'1. Adding wallclock as a feature: {wc_change:+.1f}% MAE change')
if wc_change < -5:
    print('   Wallclock is a valuable feature for the single model.')
else:
    print('   Wallclock provides limited benefit to the single model.')

if all_y_true:
    print(f'\n2. Separate experts vs single model with wallclock: {expert_vs_single:+.1f}%')
    if expert_vs_single < -5:
        print('   Mixture of experts shows meaningful improvement.')
        print('   Worth pursuing: build the routing framework.')
    elif expert_vs_single > 5:
        print('   Single model is better — separate experts hurt due to less training data.')
        print('   XGBoost already captures the wallclock signal through its tree splits.')
    else:
        print('   Results are similar — the extra complexity of separate experts')
        print('   may not be justified for this dataset and configuration.')

print(f'\n3. Note: this is a small-scale test ({SAMPLE_SIZE:,} rows, 4 windows).')
print('   A full-scale benchmark may show different results.')
print('   Also, the current model uses post-hoc features (issue #122)')
print('   which may mask the effect of wallclock-based routing.')